In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
csv_path = "/content/drive/MyDrive/Penalaran Komputer 202310370311281/tugas subcpmk3/PenalaranKomputer_CBR/04_Case_Representation/cases_representation.csv"

df = pd.read_csv(csv_path)

print(df.shape)

df.head()

(40, 10)


,case_id,filename,no_perkara,tanggal,pasal,pihak,ringkasan_fakta,argumen_hukum,length,text_full
0,1,putusan_001.txt,431 K/Pid/2026 e DEMI KEADILAN BERDASARKAN KET...,7 April 2026,Pasal 75 Undang; Pasal 433 a; Pasal 433 Ayat (...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: d g Ke...,Menimbang bahwa Putusan Pengadilan Tinggi Meda...,1994,gNomor 431 K/Pid/2026 e DEMI KEADILAN BERDASAR...
1,2,putusan_002.txt,1972 K/Pid/2025 n DEMI KnEADILAN BERDASARKAN K...,27 November 2025,Pasal 311 Ayat (1) KUHP; Pasal 253 Ayat (1) Un...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: d g Pe...,Menimbang bahwa Putusan Pengadilan Tinggi Kupa...,2588,Nomor 1972 K/Pid/2025 n DEMI KnEADILAN BERDASA...
2,3,putusan_003.txt,1805 K/Pid/2025 e DEMI KEADILAN BERDASARKAN KE...,24 Oktober 2025,Pasal 317 Ayat (1) KUHnP; Pasal 253 KUHAP; Pas...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: d g Pe...,Menimbang bahwa Putusan Pengadilan Tinggi Sura...,3112,gNomor 1805 K/Pid/2025 e DEMI KEADILAN BERDASA...
3,4,putusan_004.txt,1593 K/Pid/2025 e DEMI KEADILAN BERDASARKAN KE...,19 September 2025,Pasal 311 Ayat (1) KUHP; Pasal 310 Ayat (1) KU...,Penuntut Umum; Terdakwa; Pemohon Kasasi,didakwa dengan dakwaan sebagai berikut: o u Pe...,Menimbang bahwa Peengadilan Tinggi Jawa Tengah...,1722,gNomor 1593 K/Pid/2025 e DEMI KEADILAN BERDASA...
4,5,putusan_005.txt,1488 K/Pid/2025 e DEMI KEADILAN BERDASARKAN KE...,19 September 2025,Pasal 253 Ayat (1) KUHAP; Pasal 315 KUHP; Pasa...,Penuntut Umum; Terdakwa; Pemohon Kasasi,"didakwa dengan dakwaan Tunggal, sebagaimana di...",Menimbang bahwa putusan Pengadilan Tinggi Meda...,1537,gNomor 1488 K/Pid/2025 e DEMI KEADILAN BERDASA...


In [7]:
df["case_text"] = df["text_full"]

In [8]:
vectorizer = TfidfVectorizer(
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(
    df["text_full"]
)

In [9]:
def retrieve(query, k=5):

    query_vector = vectorizer.transform(
        [query]
    )

    similarity = cosine_similarity(
        query_vector,
        tfidf_matrix
    )[0]

    result = df.copy()

    result["similarity"] = similarity

    result = result.sort_values(
        by="similarity",
        ascending=False
    )

    return result.head(k)

In [10]:
case_solutions = {}

for _, row in df.iterrows():

    case_solutions[
        row["case_id"]
    ] = row["argumen_hukum"]

print(
    "Jumlah solusi:",
    len(case_solutions)
)

Jumlah solusi: 40


In [11]:
case_solutions[5]

'Menimbang bahwa putusan Pengadilan Tinggi Medan tersebut telah p k diberitahukan kepada Penuntut Umum pada Cabang Kejaksaan Negeri h Langkat di Pangkalan Brandean tanggal 26 Mei 2025 yang diterima melalui a Surat Tercatat pada taRnggal 3 Juni 2025 dan Penuntut Umum tersebut a i s mengajukan permohonan kasasi pada 10 Juni 2025 serta memori kasasinya M telah diterima dig Kepaniteraan Pengadilan Negeri Stabat pada tanggal 17 e Juni 2025. Dengan demikian, permohonan kasasi beserta dengan alasan- n n alasannya telah diajukan dalam tenggang waktu dan dengan cara menurut o u undang-undang, oleh karena itu permohonan kasasi Penuntut Umum d g tersebut secara formal dapat diterima; n AMenimbang bahwa alasan kasasi yang diajukan Pemohon Kasasi/Penuntut Umum dalam memori kasasi selengkapnya Itermuat dalam hberkas perkara; k a Menimbang bahwa terhadap alasan kasasi yiang diajukan Pemohon l Kasasi/Penuntut Umum tersebut, Mahkamah Agung berpendapat sebagai m b berikut: u a - Bahwa alasan kasasi Penu

In [12]:
from collections import Counter

def majority_vote_solution(top_cases):

    solutions = []

    for cid in top_cases:

        solutions.append(
            case_solutions[cid]
        )

    counter = Counter(
        solutions
    )

    return counter.most_common(1)[0][0]

In [13]:
def weighted_solution(top_result):

    best_index = top_result[
        "similarity"
    ].idxmax()

    best_case = top_result.loc[
        best_index,
        "case_id"
    ]

    return case_solutions[
        best_case
    ]

In [14]:
def predict_outcome(query):

    top_k = retrieve(
        query,
        k=5
    )

    predicted_solution = weighted_solution(
        top_k
    )

    return predicted_solution

In [15]:
query = """
Terdakwa melakukan penghinaan
dengan menyebut korban
anjing dan babi
di depan umum
"""

predict_outcome(query)

'Mahkamah Agung berpendapat bahwa a i s selaku Badan Peradilan Tertinggi yang mempunyai tugas untuk membina dan M menjaga agar semuag hukum dan undang-undang di seluruh wilayah Negara e diterapkan secara tepat dan adil, Mahkamah Agung wajib memeriksa apabila n n ada pihak yang mengajukan permohonan kasasi terhadap putusan Pengadilan o u d g n A Hal.5dari8hal. Put. No.1531K/Pid/2015 e h a R a i bawahannya yang membebaskan Terdakwa, yaitu guna menentukan sudah s M tepat dan adilka g h putusan Pengadilan bawahannya itu ; e Menimbang, bahwa alasan-alasan yang diajukan oleh Pemohon Kasasi/ n n Jaksa/Penuntut Umum pada pokoknya sebagai berikut : o u Suatu peraturan Hukum tidak diterapkan atau diterapkan tidak sebagaimana d mgestinya, yaitu sebagai berikut : 1. Bahwa Surat Dakwaan disusun Jaksa/Penuntut nUmum adalah A ALTERNATIF, yaitu Dakwaan KESATU melanggar Pasal 311 ayat (1) KUHP I atau Dakwaan KEDUA melanggar Pasal 310 ayat (1) KUHP ; h k 2. Bahwa Surat Dakwaan yang dapat dibuktikan Jaks

In [16]:
demo_queries = [

{
    "query_id":1,
    "query":"penghinaan dengan kata anjing babi",
    "actual":"Penghinaan"
},

{
    "query_id":2,
    "query":"fitnah terhadap seseorang",
    "actual":"Fitnah"
},

{
    "query_id":3,
    "query":"pencemaran nama baik",
    "actual":"Pencemaran Nama Baik"
},

{
    "query_id":4,
    "query":"penghinaan di depan umum",
    "actual":"Penghinaan"
},

{
    "query_id":5,
    "query":"merusak kehormatan seseorang",
    "actual":"Pencemaran Nama Baik"
}

]

In [17]:
results = []

for item in demo_queries:

    top_k = retrieve(
        item["query"],
        k=5
    )

    top_ids = top_k[
        "case_id"
    ].tolist()

    pred = predict_outcome(
        item["query"]
    )

    results.append({

        "query_id":
        item["query_id"],

        "query":
        item["query"],

        "actual":
        item["actual"],

        "predicted_solution":
        pred,

        "top_5_case_ids":
        str(top_ids)

    })

predictions_df = pd.DataFrame(
    results
)

predictions_df.head()

,query_id,query,actual,predicted_solution,top_5_case_ids
0,1,penghinaan dengan kata anjing babi,Penghinaan,menimbang bahwa saksi Sumah binti Casmita juga...,"[16, 34, 35, 20, 5]"
1,2,fitnah terhadap seseorang,Fitnah,Menimbang bahwa Putusan Pengadilan Tinggi Sura...,"[3, 17, 8, 27, 2]"
2,3,pencemaran nama baik,Pencemaran Nama Baik,Menimbang bahwa tindakan Terdakwa e yang menga...,"[17, 1, 40, 2, 6]"
3,4,penghinaan di depan umum,Penghinaan,Menimbang bahwa putusan Pengadilan Tinggi Meda...,"[5, 33, 13, 34, 35]"
4,5,merusak kehormatan seseorang,Pencemaran Nama Baik,Menimbang bahwa tindakan Terdakwa e yang menga...,"[17, 19, 34, 32, 18]"


In [18]:
save_path = "/content/drive/MyDrive/Penalaran Komputer 202310370311281/tugas subcpmk3/PenalaranKomputer_CBR/07_Hasil/predictions.csv"

predictions_df.to_csv(
    save_path,
    index=False
)

print(
    "predictions.csv berhasil disimpan"
)

predictions.csv berhasil disimpan
